# Bean there done that — Business & Data Understanding + KPI-analyse

**Vak:** Storytelling with Data · **Methode:** CRISP-DM
**Centrale vraag van de CEO:** *Is Bean there done that klaar om volgend jaar weer te groeien?*

**Werkwijze in dit notebook:** vóór elke analysestap staat wat we gaan doen en waarom. Na elke output volgt wat het betekent voor de beslissing van de CEO en wat de volgende stap is.

**Zo run je het:** zet `Bean_there_done_that_data.xlsx` in dezelfde map als dit notebook. Kies in VS Code rechtsboven een Python-kernel en klik op *Run All*.

In [1]:
# Run this once if a package is missing (uncomment):
# %pip install pandas openpyxl plotly nbformat

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30)

DATA_FILE = "Bean_there_done_that_data.xlsx"
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

## 1. Business Understanding

**Het bedrijf (bron: opdrachtslides)**
- Kleine Nederlandse specialty-koffiebrander: premium smaak en premium prijs, output ±1.000 ton per jaar, <1% marktaandeel, alleen verkoop in Nederland.
- 3 branding (Light, Medium, Dark), elk in een zak van 500 g of 1 kg, dus 6 producten.
- Vooral e-commerce (bezorging door PostNL, gratis verzending vanaf €55, next-day behalve in het weekend). Daarnaast een paar B2B-klanten die met een eigen vrachtwagen worden beleverd.
- Groene bonen komen via direct trade uit Costa Rica, Uganda en Colombia: zakken van 70 kg, 12 zakken per pallet, 12 pallets per 20ft-container.
- Twee branders: **Probat P60** (215 kg/u, max. batch 60 kg) en **Probat Px120** (480 kg/u, max. batch 120 kg).

**Situatie:** de groei liep de afgelopen jaren vast. Daarom is begin 2023 besloten **geen nieuwe klanten meer aan te nemen** om een stabiele situatie te krijgen. 2023 is dus ons meetjaar.

| Element | Invulling |
|---|---|
| Stakeholder | CEO van Bean there done that |
| Beslissing | Wel of niet (en zo ja, hoe) de klantenstop opheffen en groeien vanaf volgend jaar |
| Succescriterium | Per stap in de keten duidelijke KPI's met een oordeel (goed / knelpunt), plus een onderbouwd advies over de weg naar groei |
| Kosten van een verkeerde beslissing | *Te vroeg groeien:* capaciteitstekort, gemiste leverbeloftes, reputatieschade bij premiumklanten. *Te laat groeien:* gemiste omzet, en inefficiënties (bijv. bederf) die blijven bestaan |

**De keten die we analyseren:**
Inkoop → opslag grondstoffen → branden / koelen / verpakken → opslag eindproduct → levering (B2C + B2B)

**Volgende stap:** bekijken welke data we per ketenstap hebben (Data Understanding).

## 2. Data Understanding

### 2.1 Alle tabbladen inladen en de structuur bekijken
**Wat en waarom:** we laden alle tabbladen en bekijken per blad het aantal rijen en kolommen, de kolomnamen en de lege waarden. Zo koppelen we elk blad aan een stap in de keten en zien we meteen of er gaten in de data zitten.

In [2]:
sheets = pd.read_excel(DATA_FILE, sheet_name=None)

overview = pd.DataFrame({
    "rows": {k: v.shape[0] for k, v in sheets.items()},
    "columns": {k: v.shape[1] for k, v in sheets.items()},
    "missing values": {k: int(v.isna().sum().sum()) for k, v in sheets.items()},
})
overview["chain step"] = [
    "Sourcing / raw material storage",
    "Finished product storage",
    "Roasting, cooling, packing",
    "Delivery - B2C (e-commerce)",
    "Delivery - B2B",
]
overview

,rows,columns,missing values,chain step
Inventory (raw material),72,10,68,Sourcing / raw material storage
Inventory (finished product),36,8,30,Finished product storage
Roasting,36,15,0,"Roasting, cooling, packing"
E-com sales orders by product,240352,9,0,Delivery - B2C (e-commerce)
B2B orders,273,8,0,Delivery - B2B


**Interpretatie:** er zijn vijf tabbladen, en elk blad dekt precies één stap uit de keten. Daarmee kunnen we voor elke stap KPI's opstellen. Lege waarden zitten alleen in de kolom `Reasoncode` van de voorraadbladen. Dat is logisch: een reden wordt alleen ingevuld als er iets is afgeschreven. De e-commerce-data (240k orders) is verreweg de grootste bron, B2B is klein (273 regels).
**Volgende stap:** per blad de kolommen en de periode bekijken.

In [3]:
raw   = sheets["Inventory (raw material)"].copy()
fin   = sheets["Inventory (finished product)"].copy()
roast = sheets["Roasting"].copy()
ecom  = sheets["E-com sales orders by product"].copy()
b2b   = sheets["B2B orders"].copy()

for name, df in [("raw", raw), ("fin", fin), ("roast", roast), ("ecom", ecom), ("b2b", b2b)]:
    print(f"--- {name}: {df.shape}")
    print(list(df.columns))

print("\nE-com order period:", ecom["Order date"].min().date(), "->", ecom["Order date"].max().date())
print("B2B delivery period:", b2b["Delivery date"].min().date(), "->", b2b["Delivery date"].max().date())

--- raw: (72, 10)
['Product', 'Supplier', 'Month', 'Stock level', 'Unit', 'Scrapped', 'Reasoncode', 'Unit price', 'Stock value', 'Purchased']
--- fin: (36, 8)
['Product', 'Month', 'Stock level', 'Unit', 'Scrapped', 'Reasoncode', 'Unit price', 'Stock value']
--- roast: (36, 15)
['Machine', 'Month', 'Product type', 'Workdays', 'Number of batches per day', 'Load weight p/batch', 'Final weight /pbatch', 'Setup time p/batch', 'Run time p/batch (m)', 'Cooling time p/batch', 'Packing time p/batch', 'QC fail/ Weight', 'QC fail/ Color', 'QC fail/ Roast profile', 'Total rejected batches p/month']
--- ecom: (240352, 9)
['Product', 'Bag type', 'Quantity', 'Value', 'Order date', 'Delivery date', 'Delivery fee', 'Shipping costs', 'Ship to (province)']
--- b2b: (273, 8)
['Customer', 'City', 'Product', 'Bag type', 'Quantity', 'Value', 'Delivery date', 'Shipping costs']

E-com order period: 2023-01-01 -> 2023-12-31
B2B delivery period: 2023-01-02 -> 2023-12-11


**Interpretatie:** alle data gaat over kalenderjaar 2023, het eerste jaar met de klantenstop. De voorraad- en productiebladen staan per maand, de verkoopbladen per order of levering. Wat we nog niet weten: wat `Bag type` 1 en 2 betekenen. Dat is nodig om zakken om te rekenen naar kilo's.
**Volgende stap:** de betekenis van `Bag type` afleiden uit de prijs per zak.

### 2.2 Wat betekent `Bag type`?
**Wat en waarom:** we delen `Value` door `Quantity` om de prijs per zak te krijgen, en zetten die per product en bag type naast elkaar. Als bag type 2 de helft kost van bag type 1, is 1 = 1 kg en 2 = 500 g. Dat hebben we nodig om verkoop (in zakken) te vergelijken met productie en voorraad (in kg).

In [4]:
ecom["price_per_bag"] = ecom["Value"] / ecom["Quantity"]
ecom.groupby(["Product", "Bag type"])["price_per_bag"].agg(["min", "median", "max"])

min  median   max
Product      Bag type                    
Dark Roast   1        22.98   22.98 22.98
             2        11.49   11.49 11.49
Light Roast  1        26.50   26.50 26.50
             2        13.25   13.25 13.25
Medium Roast 1        25.90   25.90 25.90
             2        12.95   12.95 12.95

**Interpretatie:** de prijs per zak is vast. Bag type 2 kost precies de helft van bag type 1 (Dark €22,98 vs €11,49, Medium €25,90 vs €12,95, Light €26,50 vs €13,25). Dus **bag type 1 = 1 kg** en **bag type 2 = 500 g**. De kiloprijzen komen exact overeen met de `Unit price` in de voorraad eindproduct. Light Roast is het duurst per kg, Dark Roast het goedkoopst. Voor de CEO betekent dit dat er geen kortingen of prijsacties in de data zitten, dus omzetverschillen komen puur door volume.
**Volgende stap:** kilo's, levertijden en maandvolgorde toevoegen (Data Preparation).

## 3. Data Preparation

**Wat en waarom:** we voegen afgeleide kolommen toe die we straks voor de KPI's nodig hebben:
- verkoop in **kg** (via bag type), zodat verkoop, productie en voorraad vergelijkbaar zijn
- **levertijd in dagen** en de **weekdag van de order**, om de next-day-belofte te toetsen
- maanden als geordende categorie, zodat grafieken januari tot en met december lopen
- per brandregel het aantal batches per maand, kg in, kg uit en uren per dag
- eenduidige productnamen (in de data staat soms "Dark roast" en soms "Dark Roast")

In [5]:
BAG_KG = {1: 1.0, 2: 0.5}

# --- sales
ecom["kg"] = ecom["Quantity"] * ecom["Bag type"].map(BAG_KG)
ecom["lead_days"] = (ecom["Delivery date"] - ecom["Order date"]).dt.days
ecom["order_weekday"] = ecom["Order date"].dt.day_name()
ecom["month"] = pd.Categorical(ecom["Order date"].dt.month_name(), MONTHS, ordered=True)
ecom["Product"] = ecom["Product"].str.title()

b2b["kg"] = b2b["Quantity"] * b2b["Bag type"].map(BAG_KG)
b2b["month"] = pd.Categorical(b2b["Delivery date"].dt.month_name(), MONTHS, ordered=True)
b2b["Product"] = b2b["Product"].str.title()

# --- inventories
for df in (raw, fin):
    df["Month"] = pd.Categorical(df["Month"], MONTHS, ordered=True)
fin["Product"] = fin["Product"].str.title()

# --- roasting
roast["Month"] = pd.Categorical(roast["Month"], MONTHS, ordered=True)
roast["Product type"] = roast["Product type"].str.title()
roast["batches_month"] = roast["Number of batches per day"] * roast["Workdays"]
roast["kg_in"]  = roast["batches_month"] * roast["Load weight p/batch"]
roast["kg_out"] = roast["batches_month"] * roast["Final weight /pbatch"]
# Assumption: the roaster is occupied during setup + run time; cooling happens in a separate cooling bin.
roast["roaster_hours_per_day"] = (roast["Number of batches per day"]
                                  * (roast["Setup time p/batch"] + roast["Run time p/batch (m)"]) / 60)

print("E-com kg sold:", f"{ecom['kg'].sum():,.0f}", "| B2B kg sold:", f"{b2b['kg'].sum():,.0f}")
print("Lead time (days) range:", ecom["lead_days"].min(), "-", ecom["lead_days"].max())

E-com kg sold: 710,988 | B2B kg sold: 16,280
Lead time (days) range: 1 - 6


**Interpretatie:** na de omrekening kunnen we alles in kg vergelijken. E-commerce verkocht in 2023 ongeveer 711 ton, B2B ongeveer 16 ton, samen ~727 ton. Dat is duidelijk minder dan de "±1.000 ton" uit het bedrijfsprofiel. Dat is een eerste signaal dat het profiel niet (meer) klopt, of dat het over de inkoop van groene bonen gaat. Levertijden liggen tussen 1 en 6 dagen, er zijn geen onmogelijke waarden.
**Aanname:** voor de bezetting van de branders tellen we setup- en brandtijd, niet de koeltijd (koelen gebeurt meestal in een aparte koelbak). Noem dit in je presentatie.
**Volgende stap:** KPI's per ketenstap, beginnend bij de vraag (verkoop).

## 4. Modeling: KPI-analyse per ketenstap

We bouwen geen voorspelmodel. In deze opdracht bestaat het "model" uit een set KPI's per ketenstap, die we berekenen en visualiseren. We volgen de keten van achter naar voren: eerst de vraag, dan de productie, dan de voorraad, dan de levering.

### 4.1 Vraag — omzet per kanaal
**Wat en waarom:** we vergelijken de omzet en het volume van e-commerce en B2B. Zo weten we hoe afhankelijk het bedrijf van elk kanaal is en welk kanaal bij groei het zwaarst meetelt. Verwachting: e-commerce is veruit het grootst (dat staat ook in het profiel).

In [6]:
channel = pd.DataFrame({
    "revenue_eur": [ecom["Value"].sum(), b2b["Value"].sum()],
    "kg": [ecom["kg"].sum(), b2b["kg"].sum()],
    "orders": [len(ecom), len(b2b)],
}, index=["E-commerce", "B2B"])
channel["revenue_share_%"] = channel["revenue_eur"] / channel["revenue_eur"].sum() * 100
channel["avg_order_value_eur"] = channel["revenue_eur"] / channel["orders"]
channel

,revenue_eur,kg,orders,revenue_share_%,avg_order_value_eur
E-commerce,"17,561,219.89","710,987.50",240352,97.66,73.06
B2B,"421,652.00","16,280.00",273,2.34,"1,544.51"


**Interpretatie:** e-commerce brengt ~€17,6 mln op (~97,7% van de omzet), B2B ~€0,42 mln (~2,3%). Een gemiddelde webshoporder is ~€73, een B2B-levering ~€1.545. Voor de CEO betekent dit dat groei vooral de webshopketen belast: veel kleine pakketten, PostNL, next-day. B2B is financieel klein, maar per levering veel efficiënter.
**Volgende stap:** kijken welke producten die omzet dragen.

### 4.2 Vraag — mix van producten en verpakking
**Wat en waarom:** we berekenen per branding en zakgrootte de omzet en de kg. De mix bepaalt welke machine zwaar belast wordt (Medium gaat op de P60, Dark en Light op de Px120) en hoeveel zakken van elke maat we moeten inkopen.

In [7]:
mix = (pd.concat([ecom.assign(channel="E-commerce"), b2b.assign(channel="B2B")])
         .assign(bag=lambda d: d["Bag type"].map({1: "1 kg", 2: "500 g"}))
         .groupby(["Product", "bag"])[["Value", "kg"]].sum())
mix["kg_share_%"] = mix["kg"] / mix["kg"].sum() * 100
display(mix)

fig = px.bar(mix.reset_index(), x="Product", y="kg", color="bag", barmode="stack",
             title="Kg sold in 2023 by roast and bag size", labels={"kg": "kg sold", "bag": "Bag"})
fig.show()

Value         kg  kg_share_%
Product      bag                                      
Dark Roast   1 kg  2,223,177.12  96,744.00       13.30
             500 g 5,161,744.62 224,619.00       30.89
Light Roast  1 kg  1,129,483.00  42,622.00        5.86
             500 g 2,625,726.00  99,084.00       13.62
Medium Roast 1 kg  2,341,826.20  90,418.00       12.43
             500 g 4,500,914.95 173,780.50       23.89

**Interpretatie:** Dark Roast is het grootste product (~44% van de kg), daarna Medium (~36%) en Light (~20%). Het grootste deel van het volume gaat in zakjes van 500 g. Dat betekent meer zakken, meer verpakkingstijd en meer verpakkingsmateriaal per kg. Voor de capaciteit is belangrijk dat Dark + Light (~64% van het volume) samen op één machine draaien, de Px120.
**Volgende stap:** hoe de vraag door het jaar heen verloopt (seizoen).

### 4.3 Vraag — seizoenspatroon
**Wat en waarom:** we tellen de verkochte kg per maand en per branding. Als de vraag schommelt maar de productie niet, ontstaat voorraadopbouw (met risico op bederf) of juist tekort. Verwachting: meer koffie in de koudere maanden.

In [8]:
sales_month = (pd.concat([ecom[["month", "Product", "kg", "Value"]], b2b[["month", "Product", "kg", "Value"]]])
                 .groupby(["month", "Product"], observed=True)[["kg", "Value"]].sum().reset_index())
total_month = sales_month.groupby("month", observed=True)[["kg", "Value"]].sum()
display(total_month.T)

fig = px.line(sales_month, x="month", y="kg", color="Product", markers=True,
              title="Kg sold per month by roast (e-commerce + B2B)", labels={"kg": "kg sold", "month": ""})
fig.show()

month,January,February,March,April,May,June,July,August,September,October,November,December
kg,"63,611.00","66,175.50","52,141.00","50,822.00","52,403.50","50,942.50","52,623.00","52,232.00","70,253.00","72,711.50","70,642.00","72,710.50"
Value,"1,573,573.36","1,636,111.73","1,290,320.80","1,257,219.40","1,296,704.09","1,260,242.69","1,302,721.92","1,291,162.60","1,735,694.34","1,795,578.25","1,746,739.90","1,796,802.81"


**Interpretatie:** er is een duidelijk seizoenspatroon. Van maart t/m augustus wordt ~50 ton per maand verkocht, van september t/m februari ~65–72 ton per maand, zo'n 35–40% meer. Dat patroon zie je bij alle drie de branding. Voor de CEO betekent dit dat productie en inkoop mee moeten bewegen met het seizoen. Anders ontstaat in de zomer overschot en in de winter krapte.
**Volgende stap:** de verdeling over de regio's (relevant voor levering en eventueel B2B-groei).

### 4.4 Vraag — regio's
**Wat en waarom:** we tellen de webshopomzet per provincie. Dit laat zien waar de klantbasis zit, en waar groei (of eigen bezorging / extra B2B) het meest logisch is.

In [9]:
prov = ecom.groupby("Ship to (province)")["Value"].sum().sort_values(ascending=False).to_frame("revenue_eur")
prov["share_%"] = prov["revenue_eur"] / prov["revenue_eur"].sum() * 100
display(prov)

fig = px.bar(prov.reset_index().sort_values("revenue_eur"), x="revenue_eur", y="Ship to (province)",
             orientation="h", title="E-commerce revenue by province (2023)",
             labels={"revenue_eur": "Revenue (EUR)", "Ship to (province)": ""})
fig.show()

,revenue_eur,share_%
Ship to (province),,
Gelderland,"3,678,877.98",20.95
Noord-Brabant,"3,355,178.87",19.11
Utrecht,"2,637,750.05",15.02
Zuid-Holland,"1,935,016.38",11.02
Limburg,"1,228,291.07",6.99
Overijssel,"891,543.91",5.08
Friesland,"874,811.07",4.98
Noord-Holland,"872,153.98",4.97
Drenthe,"690,281.59",3.93


**Interpretatie:** Gelderland, Noord-Brabant en Utrecht leveren samen ongeveer 55% van de webshopomzet. Zeeland, Flevoland en Groningen zijn het kleinst. De klantbasis zit dus sterk in het midden en oosten van het land. Voor de CEO betekent dit dat er bij groei in de randprovincies het meeste ruimte is. De grote regio's zijn ook interessant voor efficiëntere bezorging.
**Volgende stap:** het B2B-kanaal in detail.

### 4.5 Vraag — B2B-klanten
**Wat en waarom:** per B2B-klant bekijken we het aantal leveringen, de omzet en de producten. Zo zien we hoe stabiel en hoe schaalbaar dit kanaal is.

In [10]:
b2b_cust = (b2b.groupby(["Customer", "City"])
               .agg(deliveries=("Value", "size"), kg=("kg", "sum"), revenue_eur=("Value", "sum"))
               .sort_values("revenue_eur", ascending=False))
display(b2b_cust)
print("Products / bag types ordered by B2B:", b2b.groupby(["Product", "Bag type"]).size().to_dict())
print("Distinct delivery days:", b2b["Delivery date"].nunique(),
      "| price per kg:", (b2b["Value"] / b2b["kg"]).unique(),
      "| shipping cost per delivery (EUR):", b2b["Shipping costs"].unique())

,,deliveries,kg,revenue_eur
Customer,City,,,
Flexplace,Utrecht,13,854.00,"22,118.60"
Nonstop Office,Utrecht,13,844.00,"21,859.60"
Desktoppers,s-Hertogenbosch,13,828.00,"21,445.20"
YourOffice,Arnhem,13,826.00,"21,393.40"
One-Stop-Office,Amersfoort,13,825.00,"21,367.50"
Premier Office,The Hague,13,819.00,"21,212.10"
The Office,Utrecht,13,818.00,"21,186.20"
Office 24/7,Utrecht,13,790.00,"20,461.00"
Rent-a-desk,Nijmegen,13,778.00,"20,150.20"


Products / bag types ordered by B2B: {('Medium Roast', 1): 273}
Distinct delivery days: 13 | price per kg: [25.9 25.9 25.9] | shipping cost per delivery (EUR): [75]


**Interpretatie:** er zijn 21 B2B-klanten, allemaal kantoor- of flexwerkplekken in de Randstad en het oosten. Elke klant krijgt 13 leveringen per jaar (elke 4 weken) en is goed voor ~€18,6k–€22,1k omzet. Alle B2B-klanten nemen alleen Medium Roast in zakken van 1 kg, tegen de gewone consumentenprijs van €25,90/kg (geen staffelkorting). Een levering met de eigen vrachtwagen kost €75. Voor de CEO is dit een zeer voorspelbaar kanaal met grote leveringen die weinig verpakkingswerk kosten. Het is een goede kandidaat om gecontroleerd te laten groeien.
**Volgende stap:** kan de productie meer vraag aan? (branden)

### 4.6 Productie — rendement en kwaliteit
**Wat en waarom:** per machine en branding berekenen we:
- **yield** = kg uit / kg in (gewichtsverlies bij het branden)
- **afkeurpercentage** = afgekeurde batches / totaal aantal batches, en de reden van afkeur
- **batch-vulgraad** = gemiddeld laadgewicht / maximale batchgrootte

Deze KPI's laten zien hoeveel groene boon er per kg verkocht product nodig is, en of er kwaliteitsproblemen zijn die bij groei groter worden.

In [11]:
MAX_BATCH = {"Probat P60": 60, "Probat Px120": 120}

prod = roast.groupby(["Machine", "Product type"]).agg(
    batches=("batches_month", "sum"),
    kg_in=("kg_in", "sum"),
    kg_out=("kg_out", "sum"),
    avg_load_kg=("Load weight p/batch", "mean"),
    rejected_batches=("Total rejected batches p/month", "sum"),
    fail_weight=("QC fail/ Weight", "sum"),
    fail_color=("QC fail/ Color", "sum"),
    fail_profile=("QC fail/ Roast profile", "sum"),
)
prod["yield_%"] = prod["kg_out"] / prod["kg_in"] * 100
prod["reject_rate_%"] = prod["rejected_batches"] / prod["batches"] * 100
prod["batch_fill_%"] = prod["avg_load_kg"] / prod.index.get_level_values("Machine").map(MAX_BATCH).values * 100
display(prod.round(1))
print(f"Total green beans roasted: {roast['kg_in'].sum():,.0f} kg -> roasted coffee: {roast['kg_out'].sum():,.0f} kg")

batches      kg_in     kg_out  avg_load_kg  \
Machine      Product type                                               
Probat P60   Medium Roast 6,090.00 333,821.10 277,784.70        54.90   
Probat Px120 Dark Roast   4,076.00 449,312.50 360,329.20       110.20   
             Light Roast  1,672.00 183,496.10 157,557.30       109.70   

                           rejected_batches  fail_weight  fail_color  \
Machine      Product type                                              
Probat P60   Medium Roast                78           20          34   
Probat Px120 Dark Roast                  63           25          18   
             Light Roast                 83           27          27   

                           fail_profile  yield_%  reject_rate_%  batch_fill_%  
Machine      Product type                                                      
Probat P60   Medium Roast            24    83.20           1.30         91.40  
Probat Px120 Dark Roast              20    80.20           1.50         91.90  
             Light Roast             29    85.90           5.00         91.40

Total green beans roasted: 966,630 kg -> roasted coffee: 795,671 kg


**Interpretatie:** in 2023 ging ~967 ton groene boon de branders in en kwam er ~796 ton gebrande koffie uit, een yield van ~80–86% (Dark heeft het meeste gewichtsverlies, zoals je verwacht bij een donkere branding). Het afkeurpercentage van **Light Roast ligt rond de 5%**, drie tot vier keer hoger dan Medium en Dark (~1,3–1,5%). Voor de CEO betekent dit dat Light Roast bij groei onevenredig veel capaciteit en bonen kost. De batch-vulgraad is al 90%+: extra kg per batch zit er nauwelijks in.
**Volgende stap:** hoeveel uren de machines per dag draaien (capaciteit).

### 4.7 Productie — capaciteitsbezetting
**Wat en waarom:** per machine berekenen we per maand hoeveel uur per werkdag de brander bezet is (batches per dag × (setup + brandtijd)). We vergelijken dat met één dienst van 8 uur. Dit is dé KPI voor de groeivraag: als de machines al vol zitten, kan er zonder investering of extra dienst niet meer geproduceerd worden.

In [12]:
util = (roast.groupby(["Month", "Machine"], observed=True)["roaster_hours_per_day"].sum()
             .unstack("Machine"))
display(util.round(2).T)
print("Average roaster hours per workday:", util.mean().round(2).to_dict())

fig = px.line(util.reset_index().melt(id_vars="Month", value_name="hours"), x="Month", y="hours",
              color="Machine", markers=True, title="Roaster occupied hours per workday (setup + run)",
              labels={"hours": "hours / workday", "Month": ""})
fig.add_hline(y=8, line_dash="dash", annotation_text="1 shift = 8 h")
fig.update_yaxes(range=[0, 10])
fig.show()

Month,January,February,March,April,May,June,July,August,September,October,November,December
Machine,,,,,,,,,,,,
Probat P60,8.00,7.64,8.06,8.31,8.36,8.16,8.02,8.11,8.03,7.93,7.88,7.78
Probat Px120,8.42,8.55,8.52,8.59,8.92,8.03,9.07,8.12,8.18,8.62,8.67,8.77


Average roaster hours per workday: {'Probat P60': 8.02, 'Probat Px120': 8.54}


**Interpretatie:** beide branders zijn gemiddeld ~8 uur per werkdag bezet (P60 ~8,0 u, Px120 ~8,5 u). In juli zat de Px120 zelfs boven de 9 uur. Met één dienst zitten de machines dus vol. En dat is in een jaar waarin er bewust níet gegroeid werd. Voor de CEO is dit de belangrijkste groeiblokkade: groei vraagt om een tweede dienst of een extra brander. Een tweede dienst verdubbelt de capaciteit zonder investering in machines. Een kanttekening: de productie ligt het hele jaar op hetzelfde niveau, terwijl de vraag schommelt (4.3).
**Volgende stap:** wat die vlakke productie doet met de voorraad eindproduct.

### 4.8 Voorraad eindproduct — productie vs. verkoop en bederf
**Wat en waarom:** per maand zetten we de productie (kg uit de brander) naast de verkoop (kg), plus de voorraad eindproduct en wat er is afgeschreven. Verwachting op basis van 4.3 en 4.7: in de zomer wordt meer geproduceerd dan verkocht, de voorraad loopt op en een deel gaat over datum.

In [13]:
prod_month = roast.groupby("Month", observed=True)["kg_out"].sum()
sales_kg_month = total_month["kg"]
fin_month = fin.groupby("Month", observed=True).agg(stock_kg=("Stock level", "sum"),
                                                     scrapped_kg=("Scrapped", "sum"))
balance = pd.DataFrame({"produced_kg": prod_month, "sold_kg": sales_kg_month}).join(fin_month)
balance["produced_minus_sold"] = balance["produced_kg"] - balance["sold_kg"]
display(balance.round(0).T)

fig = go.Figure()
fig.add_bar(x=balance.index.astype(str), y=balance["produced_kg"], name="Produced (kg)")
fig.add_bar(x=balance.index.astype(str), y=balance["sold_kg"], name="Sold (kg)")
fig.add_scatter(x=balance.index.astype(str), y=balance["stock_kg"], name="Finished stock (kg)", mode="lines+markers")
fig.update_layout(barmode="group", title="Production vs. sales vs. finished-goods stock per month")
fig.show()

,January,February,March,April,May,June,July,August,September,October,November,December
produced_kg,"68,390.00","62,419.00","71,052.00","59,929.00","65,231.00","68,029.00","66,934.00","72,256.00","67,113.00","65,648.00","69,093.00","59,578.00"
sold_kg,"63,611.00","66,176.00","52,141.00","50,822.00","52,404.00","50,942.00","52,623.00","52,232.00","70,253.00","72,712.00","70,642.00","72,710.00"
stock_kg,"35,743.00","30,548.00","47,907.00","55,555.00","66,999.00","61,334.00","59,096.00","65,560.00","60,997.00","52,752.00","50,029.00","35,460.00"
scrapped_kg,0.00,0.00,0.00,0.00,0.00,"-21,000.00","-15,000.00","-12,000.00",0.00,0.00,0.00,0.00
produced_minus_sold,"4,779.00","-3,757.00","18,911.00","9,107.00","12,828.00","17,086.00","14,311.00","20,024.00","-3,140.00","-7,063.00","-1,549.00","-13,133.00"


In [14]:
scrap = fin[fin["Scrapped"] < 0].copy()
scrap["value_eur"] = -scrap["Scrapped"] * scrap["Unit price"]
display(scrap[["Product", "Month", "Scrapped", "Reasoncode", "Unit price", "value_eur"]])
print(f"Finished product scrapped: {-scrap['Scrapped'].sum():,.0f} kg "
      f"= EUR {scrap['value_eur'].sum():,.0f} at selling price "
      f"({-scrap['Scrapped'].sum() / balance['produced_kg'].sum() * 100:.1f}% of annual production)")

,Product,Month,Scrapped,Reasoncode,Unit price,value_eur
5,Dark Roast,June,-15000,EXP,22.98,"344,700.00"
6,Dark Roast,July,-10000,EXP,22.98,"229,800.00"
7,Dark Roast,August,-5000,EXP,22.98,"114,900.00"
18,Medium Roast,July,-5000,EXP,25.90,"129,500.00"
19,Medium Roast,August,-7000,EXP,25.90,"181,300.00"
29,Light Roast,June,-6000,EXP,26.50,"159,000.00"


Finished product scrapped: 48,000 kg = EUR 1,159,200 at selling price (6.0% of annual production)


**Interpretatie:** de productie ligt elke maand rond dezelfde ~60–72 ton, terwijl de verkoop in maart t/m augustus terugzakt naar ~51–53 ton. In die maanden wordt 9–20 ton per maand méér geproduceerd dan verkocht, en de voorraad eindproduct loopt op tot ~67 ton in mei. In juni t/m augustus is **48 ton koffie afgeschreven wegens overschrijding van de houdbaarheidsdatum (EXP)**. Tegen verkoopprijs is dat ongeveer **€1,16 mln**, zo'n 6% van de jaarproductie. Voor de CEO is dit een verspilling die groei ook direct in de weg zit: de capaciteit die aan weggegooide koffie opgaat, had voor extra klanten gebruikt kunnen worden. Oplossing: productie plannen op de (seizoens)vraag, en de zomer gebruiken voor onderhoud of voorraad van snellopers.
**Volgende stap:** de andere kant van de keten, de grondstofvoorraad.

### 4.9 Voorraad grondstoffen — dekking en risico op tekorten
**Wat en waarom:** we zetten de maandvoorraad per grondstof uit en berekenen voor de groene bonen samen de **voorraaddekking in weken** (voorraad ÷ bonenverbruik van de branders per week). Een lage dekking betekent risico dat de productie stilvalt, zeker bij groei, omdat bonen in containers per zee komen en dus een lange levertijd hebben.

In [15]:
raw_pivot = raw.pivot_table(index="Month", columns="Product", values="Stock level", observed=True)
display(raw_pivot.T)

beans = raw[raw["Product"].str.startswith("Coffee Beans")]
bean_stock = beans.groupby("Month", observed=True)["Stock level"].sum()
bean_use_week = roast.groupby("Month", observed=True)["kg_in"].sum() / (52 / 12)
coverage = pd.DataFrame({"bean_stock_kg": bean_stock, "roasted_in_kg_per_week": bean_use_week})
coverage["coverage_weeks"] = coverage["bean_stock_kg"] / coverage["roasted_in_kg_per_week"]
display(coverage.round(1).T)

fig = px.line(beans, x="Month", y="Stock level", color="Product", markers=True,
              title="Green bean stock per origin (end of month, kg)", labels={"Stock level": "kg", "Month": ""})
fig.show()

print(raw.loc[raw["Purchased"] > 0].groupby("Product")["Purchased"].agg(["count", "sum", "min"]))

Month,January,February,March,April,May,June,July,August,September,October,November,December
Product,,,,,,,,,,,,
Coffee Beans (COO Colombia),"32,057.00","57,513.00","36,599.00","62,871.00","46,222.00","27,284.00","8,629.00","28,697.00","31,765.00","35,901.00","18,094.00","43,694.00"
Coffee Beans (COO Costa Rica),"24,647.00","30,076.00","13,399.00","21,858.00","26,044.00","30,627.00","15,548.00","18,910.00","4,194.00","8,192.00","11,928.00","18,949.00"
Coffee Beans (COO Uganda),"9,887.00","23,710.00","14,539.00","29,154.00","17,349.00","8,497.00","1,226.00","11,126.00","2,477.00","15,276.00","7,993.00","23,547.00"
"Coffee bag w/valve 0,5","100,000.00","80,000.00","60,000.00","50,000.00","30,000.00","110,000.00","100,000.00","80,000.00","60,000.00","50,000.00","30,000.00","20,000.00"
"Coffee bag w/valve 1,0","50,000.00","50,000.00","40,000.00","40,000.00","30,000.00","20,000.00","70,000.00","70,000.00","60,000.00","50,000.00","50,000.00","40,000.00"
Shipping box,"110,000.00","90,000.00","65,000.00","95,000.00","75,000.00","55,000.00","85,000.00","65,000.00","45,000.00","75,000.00","55,000.00","85,000.00"


Month,January,February,March,April,May,June,July,August,September,October,November,December
bean_stock_kg,"66,591.00","111,299.00","64,537.00","113,883.00","89,615.00","66,408.00","25,403.00","58,733.00","38,436.00","59,369.00","38,015.00","86,190.00"
roasted_in_kg_per_week,"19,396.30","17,597.00","20,096.30","16,526.80","18,394.20","19,312.90","18,767.90","20,222.70","18,641.30","18,431.30","18,885.10","16,796.80"
coverage_weeks,3.40,6.30,3.20,6.90,4.90,3.40,1.40,2.90,2.10,3.20,2.00,5.10


                               count     sum     min
Product                                             
Coffee Beans (COO Colombia)       12  463680   20160
Coffee Beans (COO Costa Rica)      9  181440   20160
Coffee Beans (COO Uganda)         12  342720   20160
Coffee bag w/valve 0,5             1  100000  100000
Coffee bag w/valve 1,0             1   50000   50000
Shipping box                       4  200000   50000


**Interpretatie:** de totale bonenvoorraad schommelt tussen ~1,4 en ~6,9 weken dekking. In juli zakt die tot ~1,4 week, in september en november ligt die rond de 2 weken. Per herkomst is het nog krapper: Uganda stond in juli op ~1.200 kg en in september op ~2.500 kg, Costa Rica in september op ~4.200 kg. Bonen worden ingekocht in vaste blokken van 20.160 kg (2 containers). Voor de CEO betekent dit dat het bedrijf al zonder groei dicht tegen een grondstoftekort aan zit. Bij groei moet er een veiligheidsvoorraad komen en een inkoopplanning die meebeweegt met het seizoen. Er is ook ~5 ton bonen afgeschreven op houdbaarheid (mei) en er is verpakkingsmateriaal beschadigd (DAM).
**Volgende stap:** de laatste ketenstap, de levering.

### 4.10 Levering — wordt de next-day-belofte waargemaakt?
**Wat en waarom:** we berekenen het percentage webshoporders dat de dag na bestelling wordt geleverd, en de gemiddelde levertijd per weekdag. De belofte is next-day, *behalve in het weekend*. Daarom rekenen we ook een "gecorrigeerde" versie: een vrijdag- of zaterdagorder die op maandag aankomt, telt als op tijd.

In [16]:
def promised_date(d):
    # next working day: Fri/Sat/Sun orders -> Monday
    wd = d.dayofweek
    return d + pd.Timedelta(days={4: 3, 5: 2, 6: 1}.get(wd, 1))

ecom["promised"] = ecom["Order date"].map(promised_date)
ecom["on_time"] = ecom["Delivery date"] <= ecom["promised"]

print(f"Delivered next calendar day: {(ecom['lead_days'] == 1).mean() * 100:.1f}%")
print(f"On time vs. promise (weekend-adjusted): {ecom['on_time'].mean() * 100:.1f}%")

dist = ecom["lead_days"].value_counts(normalize=True).sort_index() * 100
display(dist.to_frame("% of orders").round(1).T)

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_wd = (ecom.groupby("order_weekday")
             .agg(orders=("Value", "size"), avg_lead_days=("lead_days", "mean"), on_time_pct=("on_time", "mean"))
             .reindex(WEEKDAYS))
by_wd["on_time_pct"] *= 100
display(by_wd.round(2))

fig = px.bar(by_wd.reset_index(), x="order_weekday", y="on_time_pct",
             title="On-time delivery (weekend-adjusted) by order weekday",
             labels={"on_time_pct": "% on time", "order_weekday": ""})
fig.update_yaxes(range=[0, 100])
fig.show()

Delivered next calendar day: 63.8%
On time vs. promise (weekend-adjusted): 88.1%


lead_days,1,2,3,4,5,6
% of orders,63.80,17.90,14.30,3.20,0.70,0.10


,orders,avg_lead_days,on_time_pct
order_weekday,,,
Monday,34242,1.10,90.23
Tuesday,34242,1.08,92.12
Wednesday,34242,1.13,88.37
Thursday,34242,1.31,90.44
Friday,34242,3.18,84.90
Saturday,34242,2.17,85.05
Sunday,34900,1.17,85.30


**Interpretatie:** maar ~64% van de webshoporders komt de volgende kalenderdag aan. Na correctie voor het weekend wordt de belofte in **~88%** van de gevallen gehaald, dus ~12% van de orders (bijna 29.000 pakketten) komt te laat. Ook doordeweeks is 8–12% van de orders te laat. Vrijdag- tot en met zondagorders scoren het slechtst (~85%). De levertijd loopt op tot 6 dagen. Voor de CEO betekent dit dat de leverbelofte (een belangrijk onderdeel van de premiumpositie) nu al niet altijd wordt gehaald. Bij meer orders wordt dat waarschijnlijk erger, zeker als de productie en het verpakken (4.7) al vol zitten.
**Volgende stap:** de kosten van verzending en de regel "gratis verzending vanaf €55".

### 4.11 Levering — verzendkosten en de €55-regel
**Wat en waarom:** Bean there done that betaalt PostNL per pakket (`Shipping costs`) en rekent de klant een bezorgfee (`Delivery fee`) bij orders onder €55. We berekenen de netto verzendkosten, en we controleren of de €55-regel correct wordt toegepast.

In [17]:
ship = pd.Series({
    "shipping_costs_paid_eur": ecom["Shipping costs"].sum(),
    "delivery_fees_charged_eur": ecom["Delivery fee"].sum(),
})
ship["net_shipping_cost_eur"] = ship["shipping_costs_paid_eur"] - ship["delivery_fees_charged_eur"]
ship["net_cost_as_%_of_ecom_revenue"] = ship["net_shipping_cost_eur"] / ecom["Value"].sum() * 100
display(ship.to_frame("value"))

ecom["fee_charged"] = ecom["Delivery fee"] > 0
ecom["above_55"] = ecom["Value"] >= 55
check = pd.crosstab(ecom["above_55"], ecom["fee_charged"], margins=True)
check.index = ["Order < EUR 55", "Order >= EUR 55", "Total"]
check.columns = ["No fee", "Fee charged", "Total"]
display(check)

wrong = ecom[ecom["above_55"] & ecom["fee_charged"]]
print(f"Orders >= EUR 55 that were still charged a delivery fee: {len(wrong):,} "
      f"({len(wrong) / ecom['above_55'].sum() * 100:.1f}% of orders >= EUR 55), "
      f"EUR {wrong['Delivery fee'].sum():,.0f} charged")
print("Value range of these orders:", wrong["Value"].min(), "-", wrong["Value"].max())

,value
shipping_costs_paid_eur,"1,622,376.00"
delivery_fees_charged_eur,"599,143.05"
net_shipping_cost_eur,"1,023,232.95"
net_cost_as_%_of_ecom_revenue,5.83


,No fee,Fee charged,Total
Order < EUR 55,0,90314,90314
Order >= EUR 55,119313,30725,150038
Total,119313,121039,240352


Orders >= EUR 55 that were still charged a delivery fee: 30,725 (20.5% of orders >= EUR 55), EUR 152,089 charged
Value range of these orders: 68.94 - 106.0


**Interpretatie:** het bedrijf betaalt ~€1,62 mln aan PostNL (€6,75 per pakket) en krijgt ~€0,60 mln terug via bezorgfees. Netto kost verzending dus ~€1,0 mln, ongeveer 6% van de webshopomzet. Opvallend: **~30.700 orders van €55 of meer (20,5%) kregen tóch €4,95 bezorgkosten gerekend**, samen ~€152k. Het gaat om orders van €68,94 tot €106. Dat gaat in tegen de eigen belofte "gratis verzending vanaf €55". Het kan een fout in de webshop zijn (bijv. een drempel die ergens anders op wordt berekend) of een fout in de data. Voor de CEO is het een risico voor klanttevredenheid en reputatie. Het is een snel te controleren en te herstellen punt.
**Volgende stap:** een laatste check op de datakwaliteit, voordat we conclusies trekken.

### 4.12 Datakwaliteit — klopt het verpakkingsmateriaal met de verkoop?
**Wat en waarom:** elke verkochte zak verbruikt een koffiezak uit de grondstofvoorraad. We vergelijken het aantal verkochte zakken per maat met het verbruik volgens de voorraad (beginvoorraad + inkoop − eindvoorraad + afkeur). Zo zien we of de voorraaddata betrouwbaar is.

In [18]:
bags_sold = pd.concat([ecom, b2b]).groupby("Bag type")["Quantity"].sum().rename({1: "1 kg", 2: "500 g"})

def usage(product):
    d = raw[raw["Product"] == product].sort_values("Month")
    # start stock is derived from January: stock_jan = start + purchased - used + scrapped
    return d["Purchased"].sum() + d["Scrapped"].sum() - (d["Stock level"].iloc[-1] - d["Stock level"].iloc[0])

bag_check = pd.DataFrame({
    "bags_sold": bags_sold,
    "approx_bags_used_per_inventory": [usage("Coffee bag w/valve 1,0"), usage("Coffee bag w/valve 0,5")],
})
bag_check

,bags_sold,approx_bags_used_per_inventory
Bag type,,
1 kg,229784,60000
500 g,994967,170000


**Interpretatie:** volgens de verkoopdata zijn er ~995.000 zakken van 500 g en ~230.000 zakken van 1 kg gebruikt. Volgens de voorraaddata zijn er maar ~170.000 en ~60.000 verbruikt, ongeveer 4 à 6 keer minder. Die twee bronnen sluiten dus niet op elkaar aan. De verpakkingsvoorraad is waarschijnlijk vereenvoudigd of in een andere eenheid geregistreerd (bijv. per doos zakken). Voor de CEO en voor ons betekent dit: gebruik de verpakkingsvoorraad niet voor harde conclusies, en noem dit als beperking in de presentatie. De cijfers over bonen, productie en verkoop sluiten wel logisch op elkaar aan.
**Volgende stap:** alle KPI's samenvatten en beoordelen (Evaluation).

## 5. Evaluation — KPI-scorecard en antwoord aan de CEO

**Wat en waarom:** we zetten de belangrijkste KPI's per ketenstap in één tabel met een oordeel. Dit is de basis voor de storyline van de presentatie en voor het advies aan de CEO.

In [19]:
scorecard = pd.DataFrame([
    ("Demand", "Revenue e-commerce (EUR mln)", ecom["Value"].sum() / 1e6, "OK - dominant channel"),
    ("Demand", "Revenue B2B share (%)", channel.loc["B2B", "revenue_share_%"], "Small, stable, efficient"),
    ("Demand", "Peak vs. low season sales (x)",
        total_month["kg"].loc[["September", "October", "November", "December"]].mean()
        / total_month["kg"].loc[["April", "May", "June", "July"]].mean(), "Strong seasonality"),
    ("Roasting", "Avg roaster hours / workday P60", util["Probat P60"].mean(), "Bottleneck - full shift"),
    ("Roasting", "Avg roaster hours / workday Px120", util["Probat Px120"].mean(), "Bottleneck - full shift"),
    ("Roasting", "Reject rate Light Roast (%)", prod.xs("Light Roast", level="Product type")["reject_rate_%"].iloc[0], "Quality issue"),
    ("Finished stock", "Finished coffee scrapped (EXP, kg)", -scrap["Scrapped"].sum(), "Waste - planning issue"),
    ("Finished stock", "Value scrapped (EUR, selling price)", scrap["value_eur"].sum(), "Waste - planning issue"),
    ("Raw material", "Lowest bean coverage (weeks)", coverage["coverage_weeks"].min(), "Stock-out risk"),
    ("Delivery", "Delivered next calendar day (%)", (ecom["lead_days"] == 1).mean() * 100, "Promise under pressure"),
    ("Delivery", "On time, weekend-adjusted (%)", ecom["on_time"].mean() * 100, "Promise under pressure"),
    ("Delivery", "Net shipping cost (% of e-com revenue)", ship["net_cost_as_%_of_ecom_revenue"], "Cost driver"),
    ("Delivery", "Orders >= EUR 55 wrongly charged a fee", len(wrong), "Fix immediately"),
], columns=["Chain step", "KPI", "Value", "Assessment"])
scorecard["Value"] = scorecard["Value"].round(2)
scorecard

,Chain step,KPI,Value,Assessment
0,Demand,Revenue e-commerce (EUR mln),17.56,OK - dominant channel
1,Demand,Revenue B2B share (%),2.34,"Small, stable, efficient"
2,Demand,Peak vs. low season sales (x),1.38,Strong seasonality
3,Roasting,Avg roaster hours / workday P60,8.02,Bottleneck - full shift
4,Roasting,Avg roaster hours / workday Px120,8.54,Bottleneck - full shift
5,Roasting,Reject rate Light Roast (%),4.96,Quality issue
6,Finished stock,"Finished coffee scrapped (EXP, kg)","48,000.00",Waste - planning issue
7,Finished stock,"Value scrapped (EUR, selling price)","1,159,200.00",Waste - planning issue
8,Raw material,Lowest bean coverage (weeks),1.35,Stock-out risk
9,Delivery,Delivered next calendar day (%),63.84,Promise under pressure


**Interpretatie — antwoord aan de CEO:**
Nee, Bean there done that is **nog niet klaar om te groeien**. Zelfs in een jaar zonder nieuwe klanten:
1. **Productie zit vol.** Beide branders zijn een volle dienst bezet. Extra vraag kan alleen met een tweede dienst of een extra machine.
2. **De planning volgt de vraag niet.** De productie is vlak, de vraag seizoensgebonden. Daardoor is in de zomer 48 ton koffie (~€1,16 mln) over datum gegaan, terwijl de bonenvoorraad juist krap werd.
3. **De leverbelofte staat onder druk.** ~12% van de orders komt niet op de beloofde dag aan (ook na correctie voor het weekend), en de €55-regel voor gratis verzending wordt niet goed toegepast.

**Wel kansen:** B2B is stabiel, voorspelbaar en efficiënt (grote leveringen, één product), en de klantbasis is regionaal geconcentreerd.

**Advies (voorzet voor de presentatie):**
- *Korte termijn (verbeteren):* de fout met de €55-regel herstellen, productie plannen op de seizoensvraag (minder bederf), een veiligheidsvoorraad en inkoopritme voor de bonen afspreken, en de oorzaak van de afkeur bij Light Roast onderzoeken.
- *Voorbereiden op groei:* capaciteit uitbreiden (eerst een tweede dienst, daarna eventueel een extra brander), afspraken met PostNL over de weekend/vrijdag-levering, en groei eerst via B2B en de minder bediende provincies.

**Volgende stap:** de belangrijkste 3–4 grafieken kiezen en uitwerken volgens de storytelling-principes (context, clutter weg, aandacht sturen), en een interactief dashboard bouwen (Plotly/Dash of Streamlit).

## 6. Deployment — hoe dit bij de CEO landt

- **Presentatie (week 7):** één storyline. *"We kunnen niet groeien zolang we koffie weggooien, de machines vol zitten en we onze leverbelofte niet halen."* Daarna per knelpunt één heldere grafiek, en als afsluiter het advies.
- **Dashboard:** de KPI's uit de scorecard als een interactief dashboard (bijv. Streamlit/Dash), zodat het management maandelijks de bezetting, voorraaddekking, bederf en on-time-levering kan volgen.
- **Beperkingen die je noemt:** één jaar data (geen trend), aanname over koeltijd bij de bezetting, verpakkingsvoorraad sluit niet aan op de verkoop, geen kostprijs- en margegegevens (behalve de inkoopprijs van de bonen).